In [1]:
import re
import os
import pandas as pd
from glob import glob
paths = glob("results\*.csv") # I saved all the result csv files in one folder

In [2]:
def preprocess_eval_data(file_paths, tasks_dict):
    """
    Processes a list of CSV files and cleans model names, 
    checkpoints, and task/language metadata.
    """
    processed_dfs = []
    
    # Regex patterns for the name-paths
    # Matches 'iter_XXXX'
    # model names like ../openeurollm/revision=iter_0590000/models--openeurollm--datamix-9b-80-20/snapshots/..
    iter_pattern = re.compile(r"revision=iter_(\d+)")
    # Matches everything between 'models--' and '/snapshots'
    model_name_pattern = re.compile(r"models--(.+?)/snapshots")

    for file in file_paths:
        df = pd.read_csv(file)
        
        new_rows = []
        for _, row in df.iterrows():
            raw_model = row['model_name']
            raw_task = row['task']
            
            # Extract Checkpoint & Model Name
            checkpoint = -1 # If there is no information about checkpoint I just assign -1 to keep things easy
            model_name = raw_model
            
            if "revision=" in raw_model:
                # Extract iteration number
                # ! modify later if data format will change
                iter_match = iter_pattern.search(raw_model)
                if iter_match:
                    checkpoint = iter_match.group(1)
                
                # Extract actual model name from path
                name_match = model_name_pattern.search(raw_model)
                if name_match:
                    model_name = name_match.group(1)
            
            # geyt task and Language
            base_task_name = "unknown" # if we don't find
            language = "unknown"

            # task names can differ e.g belebele_deu_Latn, belebele_ita_Latn, global_mmlu_full_deu...
            # can't really split by "_" so I just do matching against known names
            for task_key in tasks_dict:
                if raw_task.startswith(task_key):
                    base_task_name = task_key
                    # Strip the base task name and leading underscores to get the language
                    language = raw_task.replace(task_key, "").strip("_")
                    break
            
            # Append language to model_name
            final_model_identity = f"{model_name} ({language})"

            new_rows.append({
                "model_name": final_model_identity,
                "checkpoint": checkpoint,
                "task_name": base_task_name,
                "n_shot": row['n_shot'],
                "performance": row['performance'],
                "metric": row['metric_name']
            })
            
        processed_dfs.append(pd.DataFrame(new_rows))

    return pd.concat(processed_dfs, ignore_index=True)

# Define your task prefixes
tasks_metadata = ['belebele', 'global_mmlu_full', 'include_base_44']
clean_df = preprocess_eval_data(paths,tasks_metadata)
clean_df['checkpoint'] = clean_df['checkpoint'].astype(int)
clean_df.head()

,model_name,checkpoint,task_name,n_shot,performance,metric
0,openeurollm--datamix-9b-80-20 (swe_Latn),820000,belebele,5,0.708889,"acc,none"
1,openeurollm--datamix-9b-80-20 (spa_Latn),940000,belebele,5,0.786667,"acc,none"
2,openeurollm--datamix-9b-80-20 (bul_Cyrl),360000,belebele,5,0.613333,"acc,none"
3,openeurollm--datamix-9b-80-20 (dan_Latn),520000,belebele,5,0.673333,"acc,none"
4,openeurollm--datamix-9b-80-20 (fin_Latn),840000,belebele,5,0.694444,"acc,none"


In [3]:
# for now I discard models without multiple checkpoints
# it's impossible to calculate quality metrics with only 1 checkpoint
cpt_df = clean_df[clean_df['checkpoint']!=-1]
print(len(cpt_df),len(clean_df))

6365 7504


In [29]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kendalltau

def compute_finetasks_quality_metrics(df, how='task',nr_how='simple'):
    """
    Calculates FineTasks metrics.
    how='task': Returns aggregated metrics per task (includes Ordering Consistency).
    how='model': Returns metrics for every individual model-language pair.
    """
    
    # Map base task names to number of choices - this helps to calculate non-randomness
    # add more if you eval different tasks...
    task_n_choices = {
        'belebele': 4, 'community_triviaqa': 0, 'enem': 5, 'exams': 5, 
        'faquad': 0, 'global_mmlu_full': 4, 'lumi_arc': 4, 'mlmm_arc': 4,
        'mlmm_hellaswag': 4, 'mlmm_mmlu': 4, 'oab_exams': 4, 'pawsx': 0,
        'truthfulqa': 4, 'xcodah': 4, 'xcsqa': 4, 'xstory_cloze': 2, 'xnli': 3,
        'include_base_44': 4
    }

    model_level_results = []
    task_summary_results = []

    task_groups = df.groupby('task_name')

    for task_name, task_df in task_groups:
        models_in_task = {}
        
        # Process each individual model in the task
        for model_name, model_df in task_df.groupby('model_name'):
            # Sort checkpoints!
            series = model_df.sort_values('checkpoint').set_index('checkpoint')['performance']
            
            if len(series) < 2:
                continue
                
            models_in_task[model_name] = series
            
            # Calculate metrics
            m = calculate_monotonicity(series)
            s = calculate_stn_finetasks(series)
            r = calculate_non_randomness_finetasks(series, task_name, task_n_choices,nr_how)
            
            model_level_results.append({
                "task_name": task_name,
                "model_name": model_name,
                "number_of_checkpoints": len(series),
                "monotonicity": m,
                "signal_to_noise": s,
                "non_randomness": r
            })

        # 2. If 'task' mode: aggregate the results and add ordering consistency
        if how == 'task' and models_in_task:
            task_metrics_df = pd.DataFrame([res for res in model_level_results if res['task_name'] == task_name])
            oc = calculate_ordering_consistency(models_in_task)
            
            task_summary_results.append({
                "task_name": task_name,
                "number_of_models": len(models_in_task),
                "monotonicity": task_metrics_df['monotonicity'].mean(),
                "signal_to_noise": task_metrics_df['signal_to_noise'].mean(),
                "non_randomness": task_metrics_df['non_randomness'].mean(),
                "ordering_consistency": oc
            })


    if how == 'model':
        return pd.DataFrame(model_level_results)
    else:
        return pd.DataFrame(task_summary_results)

# Quality calc functions

def calculate_monotonicity(series):
    # HF use simple Spearman correlation for monotonicty
    corr, _ = spearmanr(series.index, series.values)
    return corr

def calculate_stn_finetasks(series):
    # At least 3 checkpoints
    if len(series) < 3: return np.nan
    diffs = series.diff().abs().dropna()
    avg_std = diffs.mean()
    return series.iloc[-1] / (avg_std + 1e-8) # add small value to avoid div0!

def calculate_non_randomness_finetasks(series, task_name, task_n_choices,how='simple'):
    """
    how = simple - we take difference between highest value and baseline
    how = average_all - we calculate differences between all checkpoints and baseline, then average it
          averaging seems like a better idea to me (we know how many checkpoints perform randomly...)
    
    """
    # in original implementation non-randomness was calculated as difference between maximal and baseline score
    # using average difference of few last checkpoints might be more informative though.
    n_choices = task_n_choices.get(task_name, 0)
    baseline = 1 / n_choices if n_choices > 0 else 0.0
    if how=='simple':
        return max(0, series.max() - baseline)
    elif how=='average_all':
        mean_advantage = (series - baseline).mean()
        return mean_advantage

def calculate_ordering_consistency(models_dict, min_checkpoint=10000):
    # calculate with Kendall Tau
    common_checkpoints = set.intersection(*(set(s.index) for s in models_dict.values()))
    all_checkpoints = sorted([c for c in common_checkpoints if c >= min_checkpoint])
    
    if len(all_checkpoints) < 2 or len(models_dict) < 2:
        return np.nan

    taus = []
    for i in range(len(all_checkpoints) - 1):
        s1, s2 = all_checkpoints[i], all_checkpoints[i + 1]
        v1 = [models_dict[m].loc[s1] for m in models_dict]
        v2 = [models_dict[m].loc[s2] for m in models_dict]
        tau, _ = kendalltau(v1, v2)
        if not np.isnan(tau): taus.append(tau)

    return float(np.mean(taus)) if taus else np.nan

In [32]:
quality_df = compute_finetasks_quality_metrics(cpt_df,'task',nr_how='simple')
quality_df.head()

,task_name,number_of_models,monotonicity,signal_to_noise,non_randomness,ordering_consistency
0,belebele,24,0.968389,33.440333,0.531713,0.674977
1,global_mmlu_full,18,0.956824,53.509892,0.300357,0.890286
2,include_base_44,25,0.717734,16.593171,0.265251,0.722435


In [35]:
quality_df = compute_finetasks_quality_metrics(cpt_df,'model','simple')
quality_df.head(22)

,task_name,model_name,number_of_checkpoints,monotonicity,signal_to_noise,non_randomness
0,belebele,openeurollm--datamix-9b-80-20 (bul_Cyrl),95,0.969853,33.204439,0.525556
1,belebele,openeurollm--datamix-9b-80-20 (ces_Latn),95,0.974992,34.610655,0.527778
2,belebele,openeurollm--datamix-9b-80-20 (dan_Latn),95,0.970900,31.333321,0.537778
3,belebele,openeurollm--datamix-9b-80-20 (deu_Latn),95,0.974463,37.672529,0.557778
4,belebele,openeurollm--datamix-9b-80-20 (ell_Grek),95,0.976269,35.679940,0.522222
5,belebele,openeurollm--datamix-9b-80-20 (eng_Latn),95,0.978393,37.083447,0.595556
6,belebele,openeurollm--datamix-9b-80-20 (est_Latn),95,0.956814,29.531357,0.484444
7,belebele,openeurollm--datamix-9b-80-20 (fin_Latn),95,0.977123,37.849779,0.524444
8,belebele,openeurollm--datamix-9b-80-20 (fra_Latn),95,0.978851,36.142544,0.572222
9,belebele,openeurollm--datamix-9b-80-20 (hrv_Latn),95,0.970435,31.010297,0.534444
